# ⚙️ burst_scan — optimize

**Phase 1 searches nothing.** The config ships `n_trials: 1` and no `search:` ranges, so this notebook records one
trial of the shipped rules in the study journal, saves it as the candidate, and compares the book with buy-and-hold
— long alone, short alone and both. A search opens (phase 2) only for a sub-strategy that passed its gates in
`burst_scan_events.ipynb` (G0) and `burst_scan_model.ipynb` (a–f); read those first.

| § | what |
|---|---|
| 1 | load the panel the study scores (config `data:`) |
| 2 | every knob, long and short side by side |
| 3 | the study: topped up **to** `optuna.n_trials`, never past it, scored warm from `optuna.score_from` |
| 4 | the pick and the candidates file (`studies/<study>.candidates.yaml`) |
| 5 | **the baseline comparison** — each book against equal-weight buy-and-hold of the same names, same window |
| 6 | sensitivity — each side's detector thresholds one step either way |

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import plotly.graph_objects as go

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


import yaml

from trading_simple import optimize as opt

SUBSET = None  # a study must run on the full panel: with a SUBSET, §3 is skipped
OVERRIDES = {}  # any signals() knob (§5 and §6 only; the study runs the config)
PRM = {**strat.params, **OVERRIDES}
SCORE_FROM = strat.optuna["score_from"]
FLOOR = strat.optuna["pick_min_trades_per_year"]
INIT_CASH = settings.backtest.init_cash

## 1 · The panel the study scores

In [ ]:
panel = rs.load(settings, engine, subset=SUBSET)
span = f"{panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d}"
print(
    f"panel: {panel.close.shape[0]:,} {panel.bar} bars x {len(panel.symbols)} names ({span}); "
    f"scored from {SCORE_FROM}"
)
print("objectives:", opt.objectives_for(strat), "· study:", strat.optuna["study"])

## 2 · Every knob

A sub-strategy's own knobs carry its prefix and are shown side by side; the rest are shared. `searched` is empty
in phase 1 by design.

In [ ]:
own = sorted({k.split("_", 1)[1] for k in strat.params if k.startswith(("long_", "short_"))})
sides_tab = pd.DataFrame(
    {
        "long": {k: strat.params[f"long_{k}"] for k in own},
        "short": {k: strat.params[f"short_{k}"] for k in own},
        "doc": {k: strat.docs.get(f"long_{k}", "") for k in own},
    }
)
shared = {k: v for k, v in strat.params.items() if not k.startswith(("long_", "short_"))}
display(sides_tab)
display(pd.DataFrame({"value": pd.Series(shared), "doc": pd.Series(strat.docs)}).loc[list(shared)])
print("searched:", strat.optuna["space"] or "nothing (phase 1)")

## 3 · The study

Resumes the journal and tops it up to `optuna.n_trials` — here one trial: the shipped rules, scored from
`score_from` (the first walk-forward month; before it nothing trades). A trial runs the whole strategy, the
walk-forward included, and takes about a minute on the full panel.

In [ ]:
if SUBSET:
    print(f"SUBSET = {SUBSET}: the study journal only takes the full panel — skipped")
    study = None
else:
    try:
        done = len(opt.load_study(strat).trials)
    except KeyError:  # no journal yet
        done = 0
    todo = max(0, strat.optuna["n_trials"] - done)
    print(f"{done} trial(s) in the journal; running {todo}")
    study = opt.run_study(strat, panel, settings=settings, n_trials=todo)
    display(opt.trials_frame(study).round(4))

## 4 · The pick and the candidates file

With nothing searched the "front" is the one trial; the pick rule is the repo's (`opt.pick_from_front`, above the
`pick_min_trades_per_year` floor). The candidates file is what a phase-2 diagnostics run would compare.

In [ ]:
if study is not None:
    front = opt.pareto_frame(study)
    admissible = front[front["trades_per_year"] >= FLOOR]
    best = opt.pick_from_front(admissible if len(admissible) else front)
    print(
        f"picked trial {int(best['trial'])}: alpha {best['alpha_cagr']:+.2%}/yr, volatility "
        f"{best['volatility']:.2%}, {best['trades_per_year']:.0f} trades/yr"
        + ("" if best["beats_benchmark"] else " — NOTHING beats the hurdle (alpha > 0)")
    )
    path = opt.study_path(strat, strat.optuna["study"]).with_suffix(".candidates.yaml")
    snapshot = {
        "study": strat.optuna["study"],
        "saved": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
        "score_from": SCORE_FROM,
        "admissible": bool(len(admissible)) and bool(best["beats_benchmark"]),
        "front": [
            {
                "trial": int(r["trial"]),
                **{
                    m: round(float(r[m]), 6)
                    for m in (
                        "alpha_cagr",
                        "volatility",
                        "beta",
                        "excess_cagr",
                        "cagr",
                        "max_drawdown",
                        "trades_per_year",
                    )
                    if m in r
                },
                "params": {},
            }
            for _, r in front.iterrows()
        ],
    }
    path.write_text(yaml.safe_dump(snapshot, sort_keys=False))
    print("saved ->", path)

## 5 · The baseline comparison (Fig 5)

Every book below is scored warm from `score_from` against **equal-weight buy-and-hold of the same names over the
same window** (`strategies/baseline`, the repo's hurdle). `alpha_cagr` is the objective: a book that is flat part of
the time cannot out-earn a fully invested basket, but it must earn more than the market exposure it actually ran.
The sub-strategies are shown **alone and together**: each is judged alone first.

In [ ]:
baseline = strategies.load("baseline")


def scored(params, *, strategy=strat):
    pf, window, fees, slip = bt._run_scored(strategy, panel, params, settings, SCORE_FROM)
    st = bt.portfolio_stats(pf)
    st |= bt.active_stats(
        pf, window, fees=fees, slippage=slip, fill=strategy.fill, total_return=st["total_return"]
    )
    return pf, st


pf_base, st_base = scored({}, strategy=baseline)
books, stats = {}, {"buy & hold": st_base}
for sides in ("long", "short", "both"):
    books[sides], stats[sides] = scored({**PRM, "sides": sides})
COLS = [
    "cagr",
    "alpha_cagr",
    "beta",
    "excess_cagr",
    "volatility",
    "sharpe",
    "max_drawdown",
    "n_trades",
    "trades_per_year",
    "pct_profitable",
    "years",
]
display(pd.DataFrame(stats).loc[COLS].round(4))

fig = go.Figure()
fig.add_trace(
    plots._scatter(
        bt.equity_curve(pf_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
        name="buy & hold (equal weight)",
        line={"color": plots._MUTED, "dash": "dash", "width": 1.5},
    )
)
for sides, colr in (("long", plots._LONG), ("short", plots._SHORT), ("both", plots.SERIES[4])):
    fig.add_trace(
        plots._scatter(
            bt.equity_curve(books[sides], init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
            name=f"burst_scan · {sides}",
            line={"color": colr, "width": 1.5},
        )
    )
plots._base(
    fig, title=f"Fig 5 · cumulative return from {SCORE_FROM}, each book vs buy-and-hold", height=440
)
below(fig, -0.12)
fig.update_xaxes(type="date")
fig.update_yaxes(tickformat="+.0%")
fig.show()

## 6 · Sensitivity of the detector thresholds

Each side's two detector thresholds one step either way on the pre-registered grid (`research.GRID`), everything
else as shipped. A result that exists only at the shipped value is a spike, not a plateau. (Descriptive — the
thresholds are chosen by `burst_scan_events.ipynb` §6, never here.)

In [ ]:
def neighbours(value, ladder):
    ladder = sorted(ladder)
    i = min(range(len(ladder)), key=lambda j: abs(ladder[j] - value))
    return sorted({ladder[max(i - 1, 0)], value, ladder[min(i + 1, len(ladder) - 1)]})


sens = []
for side in SIDES:
    for knob, ladder in (("vol_mult", rs.GRID["vol_mult"]), ("range_mult", rs.GRID["range_mult"])):
        for v in neighbours(PRM[f"{side}_{knob}"], ladder):
            _, st = scored({**PRM, "sides": side, f"{side}_{knob}": v})
            sens.append(
                {
                    "side": side,
                    "knob": knob,
                    "value": v,
                    "shipped": v == PRM[f"{side}_{knob}"],
                    **{m: st[m] for m in ("alpha_cagr", "excess_cagr", "beta", "trades_per_year")},
                }
            )
sens = pd.DataFrame(sens)
sens.round(4)

## Checks

In [ ]:
rows = [
    (
        "study journal holds exactly n_trials",
        "skipped (SUBSET)" if study is None else len(study.trials),
        strat.optuna["n_trials"],
        study is None or len(study.trials) == strat.optuna["n_trials"],
    ),
    ("nothing searched in phase 1", len(strat.optuna["space"]), 0, not strat.optuna["space"]),
    (
        "baseline scored on the same window",
        f"{st_base['years']:.2f} y",
        f"{stats['both']['years']:.2f} y",
        abs(st_base["years"] - stats["both"]["years"]) < 1e-9,
    ),
]
display(rs.checks_table(rows))
rs.scorecard(
    {
        "notebook": "optimize",
        "study": strat.optuna["study"],
        "names": len(panel.symbols),
        "score_from": SCORE_FROM,
        "books": {
            k: {
                m: v[m]
                for m in (
                    "cagr",
                    "alpha_cagr",
                    "beta",
                    "excess_cagr",
                    "volatility",
                    "trades_per_year",
                )
            }
            for k, v in stats.items()
        },
    }
)